# Drain parser for BGL (Blue Gene/L) logs

Mirror of `Parser.ipynb` adapted to the LogHub **BGL** dataset.

**Format recap** (whitespace-separated, 9 fixed header fields):

```text
<LABEL> <UNIX_TS> <DATE> <NODE> <DATETIME> <NODE> <COMPONENT> <SUBCOMPONENT> <LEVEL> <MESSAGE>
```

`<LABEL>` is `-` for normal events and a short alert code (e.g. `KERNDTLB`, `APPSEV`) for anomalies.
Unlike HDFS, **anomaly labels are inline** — no separate CSV is needed.

In [1]:
import os, sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))  # project root

# ── Run tag ────────────────────────────────────────────────────
# Auto-generated as YYYYMMDD_HHMM so multiple daily runs stay distinct.
# Override before running to reload or continue a specific experiment:
#   RUN_TAG = "20260421_0800"
RUN_TAG  = datetime.now().strftime("%Y%m%d_%H%M")
RUN_DATE = RUN_TAG.split("_")[0]

# Dataset family prefix keeps BGL artefacts distinct from HDFS ones under data/processed/.
DATASET = "bgl"
ARTIFACT_PREFIX = f"{RUN_TAG}_{DATASET}"

print(f"Run tag        : {RUN_TAG}")
print(f"Dataset prefix : {DATASET}")
print(f"Outputs will be prefixed with: {ARTIFACT_PREFIX}_")


Run tag        : 20260504_2012
Dataset prefix : bgl
Outputs will be prefixed with: 20260504_2012_bgl_


In [2]:
from modules.parser import BGLParser

config_path  = "../configs/drain_bgl.ini"
bgl_log_path = "../data/raw/BGL_full.log"

n_lines = sum(1 for _ in open(bgl_log_path, "rb"))
print(f"Total lines in log: {n_lines:,}")

parser = BGLParser(config_path=config_path)

# First pass: learn templates
parser.fit_file(bgl_log_path)

# Second pass: annotate every line with its final cluster_id, template, params,
# label, node_id, component/subcomponent/level.
df = parser.annotate_file(bgl_log_path, max_lines=None)
print(df.shape)
df.head(3)

parser.export_templates(f"../data/processed/{ARTIFACT_PREFIX}_templates.json")
parser.save(f"../models/{ARTIFACT_PREFIX}_drain_parser.bin")


Total lines in log: 4,631,261
[INFO] Training Drain3 on: ../data/raw/BGL_full.log
[INFO] Processed 100000 lines...
[INFO] Processed 200000 lines...
[INFO] Processed 300000 lines...
[INFO] Processed 400000 lines...
[INFO] Processed 500000 lines...
[INFO] Processed 600000 lines...
[INFO] Processed 700000 lines...
[INFO] Processed 800000 lines...
[INFO] Processed 900000 lines...
[INFO] Processed 1000000 lines...
[INFO] Processed 1100000 lines...
[INFO] Processed 1200000 lines...
[INFO] Processed 1300000 lines...
[INFO] Processed 1400000 lines...
[INFO] Processed 1500000 lines...
[INFO] Processed 1600000 lines...
[INFO] Processed 1700000 lines...
[INFO] Processed 1800000 lines...
[INFO] Processed 1900000 lines...
[INFO] Processed 2000000 lines...
[INFO] Processed 2100000 lines...
[INFO] Processed 2200000 lines...
[INFO] Processed 2300000 lines...
[INFO] Processed 2400000 lines...
[INFO] Processed 2500000 lines...
[INFO] Processed 2600000 lines...
[INFO] Processed 2700000 lines...
[INFO] Pr

## Anomaly-rate sanity check

Per LogHub documentation, BGL has **~7.2 %** anomalous lines (332 222 of 4 631 261).

In [3]:
n_total   = len(df)
n_anomaly = int(df["is_anomaly"].sum())
n_normal  = n_total - n_anomaly

print(f"Total lines    : {n_total:,}")
print(f"Normal (-)     : {n_normal:,}  ({n_normal / n_total:.2%})")
print(f"Anomalous      : {n_anomaly:,}  ({n_anomaly / n_total:.2%})")
print()
print("Top 10 alert codes (excluding '-'):")
print(df.loc[df["is_anomaly"], "label"].value_counts().head(10))
print()
print("Log-level distribution:")
print(df["level"].value_counts())


Total lines    : 4,631,261
Normal (-)     : 4,299,039  (92.83%)
Anomalous      : 332,222  (7.17%)

Top 10 alert codes (excluding '-'):
label
KERNDTLB     152734
KERNSTOR      63491
APPSEV        41819
KERNMNTF      31531
KERNTERM      23338
KERNREC        6144
KERNRTSP       3983
APPRES         2256
APPTO          1991
KERNMICRO      1503
Name: count, dtype: int64

Log-level distribution:
level
INFO            3699499
FATAL            795015
ERROR             95017
WARNING           21446
SEVERE            18421
FAILURE            1547
Kill                306
single                4
microseconds          4
0x00544eb8,           2
Name: count, dtype: int64


## Persistence

Same fastparquet workaround as `Parser.ipynb`:
  1. JSON-encode list-valued columns (`parameters`) before writing.
  2. Cast Arrow-backed string dtypes back to plain `object` so fastparquet can serialise.

In [4]:
import json

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH  = PROCESSED_DIR / f"{ARTIFACT_PREFIX}_annotated.parquet"

df_save = df.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# fastparquet cannot write pandas Arrow-backed string columns (ArrowDtype).
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

df_save.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows  →  {PARQUET_PATH}")
print(f"File size : {PARQUET_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"Columns   : {df_save.columns.tolist()}")


Saved 4,631,261 rows  →  ../data/processed/20260504_2012_bgl_annotated.parquet
File size : 218.4 MB
Columns   : ['label', 'is_anomaly', 'unix_ts', 'date', 'node_id', 'timestamp', 'component', 'subcomponent', 'level', 'raw', 'cluster_id', 'template', 'parameters']


In [5]:
parser.validate()


[INFO] Running template validation...
[INFO] Total lines parsed   : 4631261
[INFO] Distinct templates   : 181
[OK] 100% of lines received a template assignment.
[INFO] Template support (lines per template): min=1, max=1732239, avg=25587.1
[OK] Singleton template fraction looks reasonable.
[OK] No overly generic templates detected by wildcard heuristic.
[WARN] Found 7 templates that are single-use with no wildcards (likely overfitting).
      'NULL HARDWARE SEVERE NodeCard VPD chip is not accessible'
      'NULL DISCOVERY ERROR Node card status: ALERT 0, ALERT 1, ALERT 2, ALERT 3 is (are) active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is not asserted. TEMPERATURE MASK IS ACTIVE. No temperature error. Temperature Limit Error Latch is clear. PGOOD is asserted. PGOOD error latch is clear. MPGOOD is OK. MPGOOD error latch is clear. The 2.5 volt rail is OK. The 1.5 volt rail is OK.'
      'NULL SERV_NET WARNING DeclareServiceNetworkCharacter

## Next steps

1. **Enrich templates** — reuse `Enricher.enrich_corpus_bgl` (declared in `src/enricher/enricher.py`). The output goes to `data/processed/{ARTIFACT_PREFIX}_templates_enriched.json`.
2. **Sequencer** — BGL has no native `block_id`. The two canonical options are:
   * Group by `node_id` (one sequence per compute node).
   * Slide fixed time windows over `timestamp` (e.g. 6 h), standard in DeepLog / LogBERT.
3. **PrepareDataset / GAE_Training** — same pipeline as HDFS, with the single substitution `block_id → (node_id | window_id)` as the per-graph key.